In [ ]:
# =========================================================
# INSTALL REQUIRED LIBRARIES
# =========================================================

!pip install -q timm
!pip install -q albumentations

In [ ]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import os
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm

from sklearn.metrics import accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [ ]:
# =========================================================
# REMOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# PROJECT CONFIG
# =========================================================

PROJECT_ROOT = "/content/drive/MyDrive/PersonalFashionStylistV2"

CONFIG = {

    # Paths
    "train_csv": os.path.join(
        PROJECT_ROOT,
        "datasets/metadata/train.csv"
    ),

    "val_csv": os.path.join(
        PROJECT_ROOT,
        "datasets/metadata/val.csv"
    ),

    "test_csv": os.path.join(
        PROJECT_ROOT,
        "datasets/metadata/test.csv"
    ),

    # Training
    "batch_size": 32,
    "epochs": 15,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,

    # Image
    "img_size": 224,

    # Model save
    "model_save_path": os.path.join(
        PROJECT_ROOT,
        "models/checkpoints/best_convnext_tiny.pth"
    ),

    # Seed
    "seed": 42
}

print("✅ Config initialized.")

✅ Config initialized.


In [ ]:
# =========================================================
# DEVICE SETUP
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("✅ Device:", device)

✅ Device: cuda


In [16]:
# =========================================================
# SET RANDOM SEED
# =========================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

print("✅ Random seed fixed.")

✅ Random seed fixed.


In [38]:
# =========================================================
# COPY DATASET TO LOCAL RUNTIME SSD
# =========================================================

import shutil

drive_dataset_dir = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "ecommerce",
    "dataset_clean"
)

local_dataset_dir = "/content/dataset_clean"

# Copy only once
if not os.path.exists(local_dataset_dir):

    print("🚀 Copying dataset to local SSD...")

    shutil.copytree(
        drive_dataset_dir,
        local_dataset_dir
    )

print("✅ Local dataset ready.")

🚀 Copying dataset to local SSD...
✅ Local dataset ready.


In [39]:
# =========================================================
# LOAD TRAIN / VAL / TEST CSVs
# =========================================================

train_df = pd.read_csv(CONFIG["train_csv"])
val_df = pd.read_csv(CONFIG["val_csv"])
test_df = pd.read_csv(CONFIG["test_csv"])

print("✅ Metadata loaded.\n")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

✅ Metadata loaded.

Train: 6037
Validation: 1294
Test: 1294


In [40]:
# =========================================================
# REPLACE DRIVE PATHS WITH LOCAL SSD PATHS
# =========================================================

def replace_drive_path(path):

    return path.replace(
        os.path.join(
            PROJECT_ROOT,
            "datasets",
            "ecommerce",
            "dataset_clean"
        ),
        "/content/dataset_clean"
    )

train_df["image_path"] = train_df["image_path"].apply(
    replace_drive_path
)

val_df["image_path"] = val_df["image_path"].apply(
    replace_drive_path
)

test_df["image_path"] = test_df["image_path"].apply(
    replace_drive_path
)

print("✅ Paths updated to local SSD.")

✅ Paths updated to local SSD.


In [42]:
# =========================================================
# CHECK METADATA DIRECTORY
# =========================================================

import os

metadata_dir = "/content/drive/MyDrive/PersonalFashionStylistV2/datasets/metadata"

print("✅ Files inside metadata directory:\n")

print(os.listdir(metadata_dir))

✅ Files inside metadata directory:

['train.csv', 'val.csv', 'test.csv', 'fashion_inventory_sample.csv', 'fashion_inventory.csv']


In [43]:
# =========================================================
# LOAD CLASS MAPPINGS
# =========================================================

mapping_dir = os.path.join(
    PROJECT_ROOT,
    "models",
    "label_maps"
)

with open(os.path.join(mapping_dir, "class_to_idx.json")) as f:
    CLASS_TO_IDX = json.load(f)

with open(os.path.join(mapping_dir, "idx_to_class.json")) as f:
    IDX_TO_CLASS = json.load(f)

NUM_CLASSES = len(CLASS_TO_IDX)

print("✅ Class mappings loaded.\n")

print(CLASS_TO_IDX)

✅ Class mappings loaded.

{'hoodie': 0, 'jeans': 1, 'pants': 2, 'shirt': 3, 'tshirt': 4}


In [44]:
# =========================================================
# TRAIN TRANSFORMS
# =========================================================

train_transforms = A.Compose([

    A.Resize(
        CONFIG["img_size"],
        CONFIG["img_size"]
    ),

    A.HorizontalFlip(p=0.5),

    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
        p=0.5
    ),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=10,
        p=0.5
    ),

    A.Normalize(),

    ToTensorV2()
])

print("✅ Train transforms ready.")

✅ Train transforms ready.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [45]:
# =========================================================
# VALIDATION / TEST TRANSFORMS
# =========================================================

val_transforms = A.Compose([

    A.Resize(
        CONFIG["img_size"],
        CONFIG["img_size"]
    ),

    A.Normalize(),

    ToTensorV2()
])

print("✅ Validation transforms ready.")

✅ Validation transforms ready.


In [46]:
# =========================================================
# CUSTOM DATASET CLASS
# =========================================================

class FashionDataset(Dataset):

    def __init__(self, dataframe, transforms=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["image_path"]

        label_name = row["label"]

        # Convert label → index
        label = CLASS_TO_IDX[label_name]

        # Load image
        image = Image.open(image_path).convert("RGB")

        image = np.array(image)

        # Apply transforms
        if self.transforms:

            transformed = self.transforms(image=image)

            image = transformed["image"]

        return image, label

In [47]:
# =========================================================
# CREATE DATASETS
# =========================================================

train_dataset = FashionDataset(
    train_df,
    transforms=train_transforms
)

val_dataset = FashionDataset(
    val_df,
    transforms=val_transforms
)

test_dataset = FashionDataset(
    test_df,
    transforms=val_transforms
)

print("✅ Dataset objects created.")

✅ Dataset objects created.


In [41]:
# =========================================================
# CREATE DATALOADERS
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("✅ Optimized DataLoaders created.")

✅ Optimized DataLoaders created.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [48]:
# =========================================================
# TEST DATALOADER
# =========================================================

images, labels = next(iter(train_loader))

print("✅ Batch Loaded Successfully.\n")

print("Image Tensor Shape:", images.shape)
print("Labels Shape:", labels.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


✅ Batch Loaded Successfully.

Image Tensor Shape: torch.Size([32, 3, 224, 224])
Labels Shape: torch.Size([32])


In [49]:
# =========================================================
# CREATE CONVNEXT TINY MODEL
# =========================================================

model = timm.create_model(
    "convnext_tiny",
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(device)

print("✅ ConvNeXt Tiny loaded.")

print(f"\nNumber of Classes: {NUM_CLASSES}")

✅ ConvNeXt Tiny loaded.

Number of Classes: 5


In [50]:
# =========================================================
# LOSS FUNCTION
# =========================================================

criterion = nn.CrossEntropyLoss()

print("✅ Loss function ready.")

✅ Loss function ready.


In [51]:
# =========================================================
# OPTIMIZER
# =========================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=CONFIG["learning_rate"],

    weight_decay=CONFIG["weight_decay"]
)

print("✅ AdamW optimizer initialized.")

✅ AdamW optimizer initialized.


In [52]:
# =========================================================
# LEARNING RATE SCHEDULER
# =========================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=CONFIG["epochs"]
)

print("✅ Cosine scheduler initialized.")

✅ Cosine scheduler initialized.


In [53]:
# =========================================================
# MIXED PRECISION SCALER
# =========================================================

scaler = torch.cuda.amp.GradScaler()

print("✅ Mixed precision scaler ready.")

✅ Mixed precision scaler ready.


/tmp/ipykernel_10607/1153808664.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [54]:
# =========================================================
# TRAINING FUNCTION
# =========================================================

def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with torch.cuda.amp.autocast():

            outputs = model(images)

            loss = criterion(outputs, labels)

        # Backpropagation
        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader)

    epoch_acc = accuracy_score(
        all_labels,
        all_preds
    )

    return epoch_loss, epoch_acc

In [55]:
# =========================================================
# VALIDATION FUNCTION
# =========================================================

def validate_one_epoch(model, loader):

    model.eval()

    running_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(
                preds.detach().cpu().numpy()
            )

            all_labels.extend(
                labels.detach().cpu().numpy()
            )

    epoch_loss = running_loss / len(loader)

    epoch_acc = accuracy_score(
        all_labels,
        all_preds
    )

    return epoch_loss, epoch_acc

In [56]:
# =========================================================
# TRAINING LOOP
# =========================================================

best_val_acc = 0

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

early_stop_counter = 0
early_stop_patience = 3

for epoch in range(CONFIG["epochs"]):

    print(f"\n{'='*50}")
    print(f"🚀 EPOCH {epoch+1}/{CONFIG['epochs']}")
    print(f"{'='*50}")

    # ==========================
    # TRAIN
    # ==========================

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    # ==========================
    # VALIDATE
    # ==========================

    val_loss, val_acc = validate_one_epoch(
        model,
        val_loader
    )

    # ==========================
    # SCHEDULER STEP
    # ==========================

    scheduler.step()

    # ==========================
    # SAVE HISTORY
    # ==========================

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    # ==========================
    # PRINT METRICS
    # ==========================

    print(f"\n📌 Train Loss: {train_loss:.4f}")
    print(f"📌 Train Accuracy: {train_acc:.4f}")

    print(f"\n📌 Validation Loss: {val_loss:.4f}")
    print(f"📌 Validation Accuracy: {val_acc:.4f}")

    # ==========================
    # SAVE BEST MODEL
    # ==========================

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            CONFIG["model_save_path"]
        )

        print("\n✅ Best model saved.")

        early_stop_counter = 0

    else:

        early_stop_counter += 1

    # ==========================
    # EARLY STOPPING
    # ==========================

    if early_stop_counter >= early_stop_patience:

        print("\n🛑 Early stopping triggered.")

        break


🚀 EPOCH 1/15


  0%|          | 0/189 [00:00<?, ?it/s]/tmp/ipykernel_10607/3431612964.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  0%|          | 0/41 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100%|██████████| 41/41 [00:12<00:00,  3.19it/s]



📌 Train Loss: 0.2695
📌 Train Accuracy: 0.9006

📌 Validation Loss: 0.1785
📌 Validation Accuracy: 0.9366

✅ Best model saved.

🚀 EPOCH 2/15


  0%|          | 0/189 [00:00<?, ?it/s]/tmp/ipykernel_10607/3431612964.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 41/41 [00:07<00:00,  5.32it/s]



📌 Train Loss: 0.0925
📌 Train Accuracy: 0.9672

📌 Validation Loss: 0.1347
📌 Validation Accuracy: 0.9544

✅ Best model saved.

🚀 EPOCH 3/15


...


📌 Train Loss: 0.0463
📌 Train Accuracy: 0.9839

📌 Validation Loss: 0.1483
📌 Validation Accuracy: 0.9645

🚀 EPOCH 5/15


  0%|          | 0/189 [00:00<?, ?it/s]/tmp/ipykernel_10607/3431612964.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 41/41 [00:07<00:00,  5.32it/s]



📌 Train Loss: 0.0372
📌 Train Accuracy: 0.9881

📌 Validation Loss: 0.1257
📌 Validation Accuracy: 0.9683

🚀 EPOCH 6/15


  0%|          | 0/189 [00:00<?, ?it/s]/tmp/ipykernel_10607/3431612964.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 41/41 [00:07<00:00,  5.85it/s]


📌 Train Loss: 0.0300
📌 Train Accuracy: 0.9892

📌 Validation Loss: 0.1407
📌 Validation Accuracy: 0.9629

🛑 Early stopping triggered.


In [57]:
# =========================================================
# START TRAINING
# =========================================================

print("🔥 Starting ConvNeXt Tiny Training...\n")

# Training begins automatically

🔥 Starting ConvNeXt Tiny Training...



In [58]:
# =========================================================
# SAVE TRAINING HISTORY
# =========================================================

training_history = {

    "train_losses": train_losses,
    "val_losses": val_losses,

    "train_accuracies": train_accuracies,
    "val_accuracies": val_accuracies,

    "best_val_accuracy": best_val_acc
}

history_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "convnext_tiny",
    "training_history.json"
)

with open(history_path, "w") as f:

    json.dump(training_history, f, indent=4)

print("✅ Training history saved.")
print(history_path)

✅ Training history saved.
/content/drive/MyDrive/PersonalFashionStylistV2/models/convnext_tiny/training_history.json


In [59]:
# =========================================================
# SAVE TRAINING CONFIG
# =========================================================

config_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "convnext_tiny",
    "training_config.json"
)

with open(config_path, "w") as f:

    json.dump(CONFIG, f, indent=4)

print("✅ Training config saved.")
print(config_path)

✅ Training config saved.
/content/drive/MyDrive/PersonalFashionStylistV2/models/convnext_tiny/training_config.json


In [60]:
# =========================================================
# VERIFY SAVED ARTIFACTS
# =========================================================

checkpoint_path = CONFIG["model_save_path"]

print("Checkpoint Exists:",
      os.path.exists(checkpoint_path))

print("\nCheckpoint Path:")
print(checkpoint_path)

Checkpoint Exists: True

Checkpoint Path:
/content/drive/MyDrive/PersonalFashionStylistV2/models/checkpoints/best_convnext_tiny.pth


In [61]:
# =========================================================
# SAVE FINAL MODEL SUMMARY
# =========================================================

model_summary = {

    "model_name": "convnext_tiny",

    "best_validation_accuracy": float(best_val_acc),

    "num_classes": NUM_CLASSES,

    "image_size": CONFIG["img_size"],

    "batch_size": CONFIG["batch_size"]
}

summary_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "convnext_tiny",
    "model_summary.json"
)

with open(summary_path, "w") as f:

    json.dump(model_summary, f, indent=4)

print("✅ Model summary saved.")

✅ Model summary saved.
